# P3

## Dataset as a product

In [45]:
import heapq
from collections import Counter
import json
import numpy as np

class Node:
	def __init__(self, symbol=None, frequency=None, order=0):
		self.symbol = symbol
		self.frequency = frequency
		self.order = order
		self.left = None
		self.right = None
	def __lt__(self, other):
		if self.frequency != other.frequency:
			return self.frequency < other.frequency
		return self.order < other.order

def build_huffman_tree(counts):
	heap = []

	for order, (sym, f) in enumerate(sorted(counts.items())):
		heapq.heappush(heap, Node(sym, f, order))

	order = len(heap)

	while len(heap) > 1:
		left = heapq.heappop(heap)
		right = heapq.heappop(heap)
		merged = Node(frequency=left.frequency + right.frequency, order = order)
		merged.left = left
		merged.right = right
		heapq.heappush(heap, merged)

	return heap[0]

def get_code_lengths_from_tree(root):
	lengths = {}

	def walk(node, depth=0):
		if node.symbol is not None:
			lengths[node.symbol] = depth
		else:
			walk(node.left, depth + 1)
			walk(node.right, depth + 1)

	walk(root)

	if len(lengths) == 1:
		lengths[next(iter(lengths))] = 1

	return lengths

def huffman_lengths(counts):
	if not counts:
		return {}

	root = build_huffman_tree(counts)
	return get_code_lengths_from_tree(root)

def canonical_codes(lengths):
	present = sorted(lengths, key=lambda s: (lengths[s], s))
	codes = {}
	code = 0
	prev_len = 0

	for s in present:
		L = lengths[s]
		code <<= (L - prev_len)
		codes[s] = f"{code:0{L}b}"
		code += 1
		prev_len = L

	return codes

MAGIC_BYTES = b"HC\x01"  # used by a series of bytes
MAGIC_ARRAY = b"HC\x02H" # used by numpy arrays

BASE_HEADER_BYTES = 13

def compress_bytes(data):
	if not isinstance(data, (bytes, bytearray)):
		raise TypeError("compress_bytes expects bytes")

	data = bytes(data)

	if not data:
		return MAGIC_BYTES + (0).to_bytes(8, "big") + (0).to_bytes(2, "big")

	freq = Counter(data)
	lengths = huffman_lengths(freq)
	codes = canonical_codes(lengths)

	present = sorted(freq)
	header = bytearray(MAGIC_BYTES)             
	header += len(data).to_bytes(8, "big")    
	header += len(present).to_bytes(2, "big")

	for s in present:
		header.append(s)
		header.append(lengths[s])
		
	bitstring = "".join(codes[b] for b in data)
	nbits = len(bitstring)
	payload = int(bitstring + "0" * (-nbits % 8), 2).to_bytes((nbits + 7) // 8, "big") if nbits else b""
	
	return bytes(header) + payload

def parse_header(compressed):
	compressed = bytes(compressed)

	if len(compressed) < BASE_HEADER_BYTES or compressed[:3] != MAGIC_BYTES:
		raise ValueError("bad header: wrong magic or stream too short")

	orig_len = int.from_bytes(compressed[3:11], "big")
	nums_syms = int.from_bytes(compressed[11:13], "big")

	pos = BASE_HEADER_BYTES
	present = []
	lengths = {}

	for _ in range(nums_syms):
		s = compressed[pos]
		l = compressed[pos + 1]

		if l == 0:
			raise ValueError(f"bad code length 0 for symbol {s}")

		if s in lengths:
			raise ValueError(f"duplicate symbol {s} in table")

		present.append(s)
		lengths[s] = l
		pos += 2

	return orig_len, present, lengths, pos

def decompress_bytes(compressed):
	orig_len, _, lengths, pos = parse_header(compressed)

	if orig_len == 0:
		return b""

	codes = canonical_codes(lengths)
	decoded_table = {code: sym for sym, code in codes.items()}

	out = bytearray()
	current = ""

	for byte in compressed[pos:]:
		for i in range(7, -1, -1):
			current += "1" if byte & (1 << i) else "0"

			if current in decoded_table:
				out.append(decoded_table[current])
				current = ""

				if len(out) == orig_len:
					return bytes(out)

	raise ValueError("truncated bitstream")

from functools import lru_cache

PREDICTORS = (
    "none",
    "horizontal_delta",
    "vertical_delta",
    "spiral_delta",
    "auto",
)

@lru_cache(maxsize=None)
def spiral_order(h, w):
    top, bottom, left, right = 0, h - 1, 0, w - 1
    idx = []

    while top <= bottom and left <= right:
        for x in range(left, right + 1):
            idx.append(top * w + x)

        for y in range(top + 1, bottom + 1):
            idx.append(y * w + right)

        if bottom > top:
            for x in range(right - 1, left - 1, -1):
                idx.append(bottom * w + x)

        if right > left and bottom > top:
            for y in range(bottom - 1, top, -1):
                idx.append(y * w + left)

        top += 1
        bottom -= 1
        left += 1
        right -= 1

    assert len(idx) == h * w
    order = np.array(idx, dtype=np.int64)
    order.setflags(write=False)
    return order


def is_valid(arr):
    return (
        isinstance(arr, np.ndarray)
        and arr.dtype == np.uint8
        and arr.ndim == 3
        and arr.shape[2] == 3
    )


def preprocess_delta(arr, direction="horizontal"):
    if arr.size == 0:
        return arr.copy()

    if not is_valid(arr):
        raise ValueError(
            f"Delta predictor needs uint8 HxWx3, "
            f"got {arr.dtype} {arr.shape}"
        )

    if direction == "horizontal":
        residual = np.empty_like(arr)
        residual[:, 0, :] = arr[:, 0, :]
        residual[:, 1:, :] = (
            arr[:, 1:, :].astype(np.int16)
            - arr[:, :-1, :].astype(np.int16)
        ) % 256

    elif direction == "vertical":
        residual = np.empty_like(arr)
        residual[0, :, :] = arr[0, :, :]
        residual[1:, :, :] = (
            arr[1:, :, :].astype(np.int16)
            - arr[:-1, :, :].astype(np.int16)
        ) % 256

    elif direction == "spiral":
        h, w, _ = arr.shape
        order = spiral_order(h, w)
        along = arr.reshape(-1, 3)[order]

        residual = np.empty_like(along)
        residual[0] = along[0]
        residual[1:] = (
            along[1:].astype(np.int16)
            - along[:-1].astype(np.int16)
        ) % 256

        out = np.empty((h * w, 3), dtype=np.uint8)
        out[order] = residual
        residual = out.reshape(h, w, 3)

    else:
        raise ValueError(f"Unknown predictor direction: {direction}")

    return residual.astype(np.uint8)


def postprocess_delta(residual, direction="horizontal"):
    if residual.size == 0:
        return residual.astype(np.uint8)

    recon = np.empty_like(residual)

    if direction == "horizontal":
        recon[:, 0, :] = residual[:, 0, :]
        for x in range(1, residual.shape[1]):
            recon[:, x, :] = (
                residual[:, x, :].astype(np.int16)
                + recon[:, x - 1, :].astype(np.int16)
            ) % 256

    elif direction == "vertical":
        recon[0, :, :] = residual[0, :, :]
        for y in range(1, residual.shape[0]):
            recon[y, :, :] = (
                residual[y, :, :].astype(np.int16)
                + recon[y - 1, :, :].astype(np.int16)
            ) % 256

    elif direction == "spiral":
        h, w, _ = residual.shape
        order = spiral_order(h, w)
        along = residual.reshape(-1, 3)[order]

        rec = np.cumsum(along.astype(np.int64), axis=0) % 256

        out = np.empty((h * w, 3), dtype=np.uint8)
        out[order] = rec
        recon = out.reshape(h, w, 3)

    else:
        raise ValueError(f"Unknown predictor direction: {direction}")

    return recon.astype(np.uint8)


def encode_with_predictor(arr, predictor):
    if predictor != "none" and not is_valid(arr):
        predictor = "none"

    if predictor == "horizontal_delta":
        encoded_arr = preprocess_delta(arr, "horizontal")
    elif predictor == "vertical_delta":
        encoded_arr = preprocess_delta(arr, "vertical")
    elif predictor == "spiral_delta":
        encoded_arr = preprocess_delta(arr, "spiral")
    else:
        encoded_arr = arr

    meta = {
        "shape": list(arr.shape),
        "dtype": str(arr.dtype),
        "predictor": predictor,
    }
    meta_bytes = json.dumps(
        meta, separators=(",", ":")
    ).encode("utf-8")

    if len(meta_bytes) > 0xFFFFFFFF:
        raise ValueError("Array metadata is too large")

    raw_bytes = np.ascontiguousarray(encoded_arr).tobytes()
    payload = compress_bytes(raw_bytes)

    return (
        MAGIC_ARRAY
        + len(meta_bytes).to_bytes(4, "big")
        + meta_bytes
        + payload
    )


def compress_array(arr, settings=None):
    if not isinstance(arr, np.ndarray):
        raise TypeError("compress_array expects a np.ndarray")

    settings = settings or {}
    predictor = settings.get("predictor", "auto")

    if predictor not in PREDICTORS:
        raise ValueError(f"Unknown predictor: {predictor}. Valid: {PREDICTORS}")

    if predictor == "auto":
        if not is_valid(arr):
            return encode_with_predictor(arr, "none")

        candidates = (
            encode_with_predictor(arr, "none"),
            encode_with_predictor(arr, "spiral_delta"),
            encode_with_predictor(arr, "vertical_delta"),
            encode_with_predictor(arr, "horizontal_delta"),
        )

        return min(candidates, key=len)

    return encode_with_predictor(arr, predictor)


def decompress_array(compressed):
    if len(compressed) < 4:
        raise ValueError("Truncated array header")

    meta_len = int.from_bytes(compressed[:4], "big")
    meta_end = 4 + meta_len

    if meta_end > len(compressed):
        raise ValueError("Truncated array metadata")

    try:
        meta = json.loads(compressed[4:meta_end].decode("utf-8"))
        shape = tuple(meta["shape"])
        dtype = np.dtype(meta["dtype"])
        predictor = meta["predictor"]
    except (UnicodeDecodeError, json.JSONDecodeError, KeyError,
            TypeError, ValueError) as exc:
        raise ValueError("Invalid array metadata") from exc

    if predictor not in PREDICTORS or predictor == "auto":
        raise ValueError(f"Invalid stored predictor: {predictor}")

    if any(not isinstance(d, int) or d < 0 for d in shape):
        raise ValueError("Invalid array shape")

    raw_bytes = decompress_bytes(compressed[meta_end:])

    expected_nbytes = int(np.prod(shape, dtype=object)) * dtype.itemsize
    if len(raw_bytes) != expected_nbytes:
        raise ValueError(
            f"Decoded byte count mismatch: expected {expected_nbytes}, "
            f"got {len(raw_bytes)}"
        )

    arr = np.frombuffer(raw_bytes, dtype=dtype).copy().reshape(shape)

    if predictor != "none":
        if not is_valid(arr):
            raise ValueError("Stored predictor is incompatible with array")

        direction = {
            "horizontal_delta": "horizontal",
            "vertical_delta": "vertical",
            "spiral_delta": "spiral",
        }[predictor]

        arr = postprocess_delta(arr, direction)

    return arr

def compress(data, settings=None):
	if isinstance(data, np.ndarray):
		return compress_array(data, settings)
	return compress_bytes(data)

def decompress(compressed):
	if compressed[:len(MAGIC_ARRAY)] == MAGIC_ARRAY:
		return decompress_array(compressed[4:])
	if compressed[:len(MAGIC_BYTES)] == MAGIC_BYTES:
		return decompress_bytes(compressed)
	raise ValueError("unknown format: " + repr(compressed[:4]))

from pathlib import Path
RESOURCE_DIR_KIRMIZI = Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio")

def load_images(folder: Path, limit=None):
    records = []
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        if limit is not None and len(records) >= limit:
            break
        with Image.open(path) as im:
            arr = np.asarray(im.convert("RGB"))
        records.append({"path": path, "arr": arr})
    return records

In [46]:
import hashlib, json, logging, os, random, time, tracemalloc
from pathlib import Path
import numpy as np
from PIL import Image

CONFIG = {
    "seed": 42,
    "raw_dir": Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio"),
    "out_dir": Path("data/processed"),
    "manifest_path": Path("data/manifest.json"),
    "expected_shape": (600, 600, 3),
    "expected_dtype": np.dtype("uint8"),
    "valid_min": 0, "valid_max": 255,
    "predictor": "auto",
}
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("p3")

In [47]:
def validate(arr):
    if not isinstance(arr, np.ndarray):
        raise TypeError(f"Expected ndarray, got {type(arr).__name__}")
    if arr.shape != CONFIG["expected_shape"]:
        raise ValueError(f"Invalid shape {arr.shape}, expected {CONFIG['expected_shape']}")
    if arr.dtype != CONFIG["expected_dtype"]:
        raise ValueError(f"Invalid dtype {arr.dtype}, expected {CONFIG['expected_dtype']}")
    if (arr < CONFIG["valid_min"]).any() or (arr > CONFIG["valid_max"]).any():
        raise ValueError(f"Values outside [{CONFIG['valid_min']}, {CONFIG['valid_max']}]")

def load_raw(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")
    with Image.open(path) as im:
        arr = np.asarray(im.convert("RGB"))
    validate(arr)
    return arr

def write_atomic(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as fh:
        fh.write(data); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path) 

In [ ]:
def load_images(folder: Path, limit=None):
    records = []
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        if limit is not None and len(records) >= limit:
            break
        try:
            arr = load_raw(path)
            records.append({"path": path, "arr": arr})
        except Exception as e:
            log.error(f"Skipping {path}: {e}")
    return records

In [58]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def make_manifest(record_list):
    manifest = []
    for rec in record_list:
        src = rec["path"]
        out_path = CONFIG["out_dir"] / (src.stem + ".hcnp")  # custom extension
        manifest.append({
            "original": str(src.relative_to(CONFIG["raw_dir"])),
            "processed": str(out_path.relative_to(CONFIG["out_dir"])),
            "size_original": src.stat().st_size,
            "size_processed": out_path.stat().st_size,
            "sha256_original": sha256_file(src),
            "sha256_processed": sha256_file(out_path),
        })
    with open(CONFIG["manifest_path"], "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

In [59]:
LIMIT = 10

start = time.time()
records = load_images(CONFIG["raw_dir"], limit=LIMIT, validate=True)  # change limit to None for full dataset
log.info(f"Loaded {len(records)} images in {time.time()-start:.2f}s")

# Compress each, save, record timing and memory
timing = []
for rec in records:
    t0 = time.time()
    compressed = compress_array(rec["arr"], {"predictor": CONFIG["predictor"]})
    out_path = CONFIG["out_dir"] / (rec["path"].stem + ".hcnp") # custom file extension
    write_atomic(out_path, compressed)
    dt = time.time() - t0
    timing.append({"path": str(rec["path"]), "time_s": dt, "size_kb": len(compressed) / 1024})
    log.info(f"Compressed {rec['path'].name} -> {out_path.name} ({dt:.3f}s, {len(compressed)/1024:.2f} KB)")

# Save timing evidence
with open(CONFIG["out_dir"] / "timing.json", "w") as f:
    json.dump(timing, f, indent=2)

2026-11-04 14:26:11,896 INFO Loaded 10 images in 0.05s
2026-11-04 14:26:12,492 INFO Compressed kirmizi (1).jpg -> kirmizi (1).hcnp (0.594s, 225.24 KB)
2026-11-04 14:26:13,034 INFO Compressed kirmizi (10).jpg -> kirmizi (10).hcnp (0.541s, 216.52 KB)
2026-11-04 14:26:13,587 INFO Compressed kirmizi (11).jpg -> kirmizi (11).hcnp (0.552s, 235.48 KB)
2026-11-04 14:26:14,214 INFO Compressed kirmizi (12).jpg -> kirmizi (12).hcnp (0.626s, 207.27 KB)
2026-11-04 14:26:14,878 INFO Compressed kirmizi (13).jpg -> kirmizi (13).hcnp (0.664s, 229.47 KB)
2026-11-04 14:26:15,494 INFO Compressed kirmizi (14).jpg -> kirmizi (14).hcnp (0.616s, 207.31 KB)
2026-11-04 14:26:16,058 INFO Compressed kirmizi (15).jpg -> kirmizi (15).hcnp (0.563s, 241.57 KB)
2026-11-04 14:26:16,614 INFO Compressed kirmizi (16).jpg -> kirmizi (16).hcnp (0.555s, 200.62 KB)
2026-11-04 14:26:17,170 INFO Compressed kirmizi (17).jpg -> kirmizi (17).hcnp (0.555s, 228.51 KB)
2026-11-04 14:26:17,713 INFO Compressed kirmizi (18).jpg -> kirmi

In [ ]:
def verify(record):
    original = record["arr"]
    processed_path = CONFIG["out_dir"] / (record["path"].stem + ".hcnp")
    with open(processed_path, "rb") as f:
        compressed = f.read()
    recon = decompress(compressed)
    assert np.array_equal(original, recon), "Round‑trip mismatch!"
    return True

errors = []
for rec in records[:LIMIT]:  # test first few
    try:
        assert verify(rec)
        log.info(f"Verified {rec['path'].name}")
    except Exception as e:
        errors.append(str(rec["path"]) + ": " + str(e))
        log.error(f"Verification failed for {rec['path']}: {e}")
if errors:
    log.warning(f"{len(errors)} verification errors")

2026-11-04 14:24:13,955 INFO Verified kirmizi (1).jpg
2026-11-04 14:24:14,373 INFO Verified kirmizi (10).jpg
2026-11-04 14:24:14,767 INFO Verified kirmizi (11).jpg
2026-11-04 14:24:15,127 INFO Verified kirmizi (12).jpg
2026-11-04 14:24:15,510 INFO Verified kirmizi (13).jpg
2026-11-04 14:24:15,854 INFO Verified kirmizi (14).jpg
2026-11-04 14:24:16,268 INFO Verified kirmizi (15).jpg
2026-11-04 14:24:16,586 INFO Verified kirmizi (16).jpg
2026-11-04 14:24:16,950 INFO Verified kirmizi (17).jpg
2026-11-04 14:24:17,298 INFO Verified kirmizi (18).jpg


In [56]:
try:
    load_raw(Path("nonexistent.jpg"))
except FileNotFoundError as e:
    log.warning(f"Caught expected file error: {e}")

try:
    validate(np.zeros((3,3,3), dtype=np.float32))
except ValueError as e:
    log.warning(f"Caught expected validation error: {e}")

2026-11-04 14:24:19,074 WARNING Caught expected file error: Missing input: nonexistent.jpg
2026-11-04 14:24:19,075 WARNING Caught expected validation error: Invalid shape (3, 3, 3), expected (600, 600, 3)


In [57]:
# Cell 10: measure memory used by a single image compression
tracemalloc.start()
rec = records[0]
_ = compress_array(rec["arr"], {"predictor": "auto"})
current, peak = tracemalloc.get_traced_memory()
print(f"Peak memory during compression: {peak / 1e6:.2f} MB")
tracemalloc.stop()

Peak memory during compression: 14.56 MB
